# M2 parsing benchmark: the whole corpus

Parses the **20,000 assigned papers** (every version) of the five crawl runs into the `elements`, `hierarchy` and `bibtex` Parquet tables on a Colab **CPU** runtime, and measures wall time, RAM and disk. It does not crawl anything.

1. Preflight reads the five runs on Drive and stops unless the crawl is complete (20,000 IDs, no incomplete chunk).
2. Each run is restored from its Drive checkpoint to the local disk **with SHA-256 verification**. This copy is setup and is outside the benchmark.
3. `src.benchmark.m2_benchmark` runs `src.parser.run` over all five runs as one corpus (`--expect-assigned`) with the resource sampler.
4. The state is copied to Drive as the run goes. If the runtime dies, open the notebook on a new runtime, set `RESUME = True` with the **same** `M2_RUN_ID`, and run again: only the work after the last committed shard is repeated. The report keeps the log of every session, as the lecturer asked for multi-session benchmarks.

In [ ]:
# Directory configuration: verify these paths for the mounted Drive.
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
RUNS = {  # run id -> folder of the run below DRIVE_DIR (each holds data/corpus and work/)
    "only-v1-all": "only-v1/runs/only-v1-all",
    "multi-dang-0000-4999": "multi-version/runs/multi-dang-0000-4999",
    "multi-hieu-0000-4999": "multi-version/runs/multi-hieu-0000-4999",
    "multi-thien-0000-4999": "multi-version/runs/multi-thien-0000-4999",
    "multi-tien-0000-4999": "multi-version/runs/multi-tien-0000-4999",
}
M2_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m2"  # results and resume state go here
LOCAL_ROOT = "/content/m2-local"  # the restored corpus, on the local disk of the runtime
BBL_RECOVERY_DIR = (
    "/content/drive/MyDrive/bbl-recovery-kit/results/run-01"  # Separate input; adjust if needed
)

In [ ]:
# Edit this cell. Use a NEW id for a new experiment, the SAME id (and RESUME = True) to resume.
RUN_MODE = "full"  # Parse all assigned papers; "smoke" remains available for a sample
SMOKE_LIMIT = 1000  # First 1,000 papers in stable ID order
SMOKE_OFFSET = 0  # Zero-based offset into the ordered corpus
M2_RUN_ID = "m2-full-01"  # Use a fresh, distinct run ID for each new benchmark
RESUME = False  # True on a new runtime after the session of this M2_RUN_ID was lost
PREVIOUS_SESSION_STOPPED = False  # Set True only after verifying the previous runtime has stopped
WORKERS = 2  # the plan: at most 2 processes, under about 4 GB each
SHARD_PAPERS = 100  # papers per Parquet shard (bounds the memory of the parent process)
PAPER_TIMEOUT_S = 600  # a paper that takes longer becomes an 'error' row instead of stalling
RESTORE_WORKERS = 8  # Concurrent Drive reads, bounded to 1-32
REUSE_PARTIAL_RESTORE = False  # Enable only after the prior runtime has stopped
SYNC_EVERY_S = 600  # copy the run state to Drive this often
EXPECTED_IDS = 20000
RUN_M1_STATS = False  # M1 export requires the full M1 checkpoint, not M2-only restore
RUN_SCAN = False  # True: gather + references only (a first look at the real corpus)
RUN_M2 = False  # True only after the preflight has been checked
CODE_REF = "main"  # New runs resolve main to an exact SHA; resume uses the saved SHA
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"

In [ ]:
# Prevent a new checkout from being mislabeled while Python keeps old src modules.
import sys

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError(
        "Restart runtime before setup; cached pipeline modules may belong to another revision."
    )

## Authorize Google Drive

Use the Google account that can read the five run folders (add them to My Drive as shortcuts if they are shared) and write `M2_DRIVE_DIR`.

In [ ]:
import json
import os
import shutil
import subprocess
import time

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
if not DRIVE.is_dir():
    raise FileNotFoundError("Check DRIVE_DIR: the team folder is not visible in this Drive")
M2_DRIVE = Path(M2_DRIVE_DIR) / M2_RUN_ID
M2_DRIVE.mkdir(parents=True, exist_ok=True)

In [ ]:
revision = CODE_REF
saved_path = M2_DRIVE / "M2_MANIFEST.json"
if RESUME:
    revision = json.loads(saved_path.read_text())["code_revision"]
elif saved_path.exists():
    raise FileExistsError(f"{saved_path} exists: use a new M2_RUN_ID, or RESUME = True")

In [ ]:
repo_dir = Path("/content/CodeRepo")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Implementation revision:", REVISION)

## Install dependencies

Package installation is setup and is outside the benchmark.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "psutil", "pyarrow", "pandas", "tqdm"],
    check=True,
)

## Preflight: is the crawl complete?

Reads `MANIFEST.json` and `work/coverage.json` of each run. The run stops here unless every run is complete, no chunk is incomplete, and the runs together cover exactly the assigned IDs. The separate BBL recovery input is verified by the parser. Recovery costs do not belong to the M1 crawl benchmark.

In [ ]:
import pandas as pd

rows = []
for run_id, rel in RUNS.items():
    base = DRIVE / rel
    for name in ("INDEX.json", "MANIFEST.json", "work/coverage.json"):
        if not (base / name).is_file():
            raise FileNotFoundError(f"{base / name} is missing")
    manifest = json.loads((base / "MANIFEST.json").read_text())
    coverage = json.loads((base / "work/coverage.json").read_text())
    rows.append(
        {
            "run": run_id,
            "required_ids": coverage["required_ids"],
            "source_complete": coverage["papers_source_complete"],
            "complete": coverage["complete"],
            "incomplete_chunks": len(coverage["incomplete_chunks"]),
            "source_mode": coverage["source_mode"],
            "revision": manifest["code_revision"][:7],
        }
    )
preflight = pd.DataFrame(rows).set_index("run")
display(preflight)
problems = []
if preflight["required_ids"].sum() != EXPECTED_IDS:
    problems.append(f"{preflight['required_ids'].sum()} IDs in the runs, expected {EXPECTED_IDS}")
if (preflight["source_complete"] != preflight["required_ids"]).any():
    problems.append("a run has papers whose source is not complete")
if (~preflight["complete"]).any() or preflight["incomplete_chunks"].any():
    problems.append("a run is not complete or has incomplete chunks")
if problems:
    raise RuntimeError("; ".join(problems))
print("Crawl complete:", int(preflight["required_ids"].sum()), "papers in", len(RUNS), "runs")

## Record the configuration

A new run writes `M2_MANIFEST.json` to Drive; a resume must use the same revision and settings.

In [ ]:
import hashlib

bbl_dir = Path(BBL_RECOVERY_DIR)
if not (bbl_dir / "versions.jsonl").is_file():
    raise FileNotFoundError(f"Recovery journal missing: {bbl_dir}")


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be smoke or full")
if RUN_MODE == "smoke" and (
    SMOKE_LIMIT < 1 or SMOKE_OFFSET < 0 or SMOKE_OFFSET + SMOKE_LIMIT > EXPECTED_IDS
):
    raise ValueError("Smoke offset and limit must select papers within the assigned corpus")
if RUN_MODE == "full" and "smoke" in M2_RUN_ID.lower():
    raise ValueError("Use a distinct full-run M2_RUN_ID")

manifest = {
    "m2_run_id": M2_RUN_ID,
    "run_mode": RUN_MODE,
    "smoke_limit": SMOKE_LIMIT if RUN_MODE == "smoke" else None,
    "smoke_offset": SMOKE_OFFSET if RUN_MODE == "smoke" else None,
    "code_revision": REVISION,
    "runs": RUNS,
    "bbl_journal_sha256": sha256_file(bbl_dir / "versions.jsonl"),
    "crawl_index_sha256": {
        name: sha256_file(DRIVE / rel / "INDEX.json") for name, rel in RUNS.items()
    },
    "workers": WORKERS,
    "shard_papers": SHARD_PAPERS,
    "paper_timeout_s": PAPER_TIMEOUT_S,
    "expected_ids": EXPECTED_IDS,
    "crawl_revisions": sorted(set(preflight["revision"])),
}
if RESUME:
    saved = json.loads(saved_path.read_text())
    for key in (
        "workers",
        "shard_papers",
        "paper_timeout_s",
        "expected_ids",
        "run_mode",
        "smoke_limit",
        "smoke_offset",
        "runs",
        "bbl_journal_sha256",
        "crawl_index_sha256",
    ):
        if saved[key] != manifest[key]:
            raise ValueError(f"RESUME needs the same {key}: saved {saved[key]!r}")
else:
    saved_path.write_text(json.dumps(manifest, indent=2) + "\n")

## Restore the five corpora to local disk (setup, not benchmarked)

The M2 restore copies only corpus files and `work/versions.jsonl`. It reads each Drive source once while copying and checking SHA-256, then verifies the local bytes. A bounded worker pool and file-level progress make the operation observable; Drive speed and throttling vary. Existing partial files are reused only when `REUSE_PARTIAL_RESTORE = True` and their local SHA-256 matches the pinned index. Never reuse a folder while another runtime is writing it. The `.bbl` recovery stays in its separate Drive folder.


In [ ]:
from tqdm.auto import tqdm

from src.benchmark.m2_restore import restore_m2

started = time.perf_counter()
DATA_ROOTS, CRAWL_WORK_DIRS = [], []
for run_id, rel in RUNS.items():
    print(f"Preparing {run_id}: checking and restoring M2 inputs...", flush=True)
    local = Path(LOCAL_ROOT) / run_id
    index_path = DRIVE / rel / "INDEX.json"
    index_bytes = index_path.read_bytes()
    expected_index = manifest["crawl_index_sha256"][run_id]
    if hashlib.sha256(index_bytes).hexdigest() != expected_index:
        raise ValueError(f"Checkpoint index changed since preflight: {run_id}")
    pinned_index = json.loads(index_bytes)
    total = sum(
        name.startswith("data/corpus/") or name == "work/versions.jsonl"
        for name in pinned_index["files"]
    )
    with tqdm(total=total, desc=f"Restore {run_id}", unit="file", mininterval=0) as bar:

        def show_progress(event):
            pending = event["completed"] - bar.n
            if pending >= 25 or event["completed"] == event["total"]:
                bar.update(pending)

        result = restore_m2(
            local,
            DRIVE / rel,
            pinned_index,
            workers=RESTORE_WORKERS,
            reuse_partial=REUSE_PARTIAL_RESTORE,
            on_progress=show_progress,
            on_phase=lambda phase: tqdm.write(f"Restore phase: {phase}"),
            expected_index_sha256=expected_index,
        )
    print(run_id, "restored:", result)
    DATA_ROOTS.append(local / "data" / "corpus")
    CRAWL_WORK_DIRS.append(local / "work")
RESTORE_SECONDS = time.perf_counter() - started
print(f"Restore (setup, outside the benchmark): {RESTORE_SECONDS / 60:.1f} min")
subprocess.run(["df", "-h", "/content"], check=True)

In [ ]:
from src.benchmark.m2_sync import restore_from_drive, sync_to_drive

OUT = Path("/content/m2-out") / M2_RUN_ID
WORK = Path("/content/m2-work") / M2_RUN_ID
REPORT = Path("/content/m2-report") / M2_RUN_ID
if RESUME:
    print("Restored from Drive:", restore_from_drive(M2_DRIVE, OUT, WORK, REPORT))

## Optional first look: gather and references only

Set `RUN_SCAN = True` to see, without writing tables, how many versions are parsed, empty or without a root, and where the parser would have trouble.

In [ ]:
if RUN_SCAN:
    cmd = [
        sys.executable,
        "-m",
        "src.parser.run",
        "--scan",
        "--out-dir",
        "/content/m2-scan-out",
        "--work-dir",
        "/content/m2-scan-work",
    ]
    cmd += ["--bbl-recovery-dir", str(bbl_dir)]
    for root in DATA_ROOTS:
        cmd += ["--data-root", str(root)]
    for work in CRAWL_WORK_DIRS:
        cmd += ["--crawl-work-dir", str(work)]
    if RUN_MODE == "smoke":
        cmd += ["--offset", str(SMOKE_OFFSET), "--limit", str(SMOKE_LIMIT)]
    subprocess.run(cmd + ["--workers", str(WORKERS), "--expect-assigned"], check=True)
    display(pd.read_csv("/content/m2-scan-work/m2_versions.csv")["status"].value_counts())

## Parse and benchmark

One command for the whole corpus, resumable. The sampler measures the parser and its worker processes; the copy to Drive runs in this notebook every `SYNC_EVERY_S` seconds and is not part of the parser, but it competes for the disk and the network, so the wall time is that of a run that also saves its state. Set `RUN_M2 = True` to start.

Before the full run, complete a Colab smoke test and interruption/resume check with both inputs; this notebook does not certify a run automatically.


After an interrupted session, first ensure the previous runtime has stopped, then set `PREVIOUS_SESSION_STOPPED = True`. Missing interrupted-session measurements are reported as lower bounds. Never run two sessions against the same M2 run ID.

The measured parser process verifies the restored corpus and recovered BBL hashes before parsing; this verification is included in parser wall time. Restoring the corpus from Drive is setup and is measured separately. Repeated shard integrity checks during Drive sync consume resources and can affect parser wall time.

The default smoke mode parses the first 1,000 papers in stable ID order with a separate run ID. The progress bar updates every 25 papers and includes already committed papers on resume. Preflight and corpus restore happen before the parsing progress bar. Switch to full mode and a new run ID only after checking smoke and resume. Smoke figures are not an official benchmark.


In [ ]:
if RUN_M2:
    from src.benchmark.m2_session_state import annotate_report, begin_session, finish_session

    session_state = M2_DRIVE / "SESSION_STATE.json"
    session_id = begin_session(session_state, previous_session_stopped=PREVIOUS_SESSION_STOPPED)
    cmd = [
        sys.executable,
        "-m",
        "src.benchmark.m2_benchmark",
        "--out-dir",
        str(OUT),
        "--work-dir",
        str(WORK),
        "--report",
        str(REPORT),
    ]
    for root in DATA_ROOTS:
        cmd += ["--data-root", str(root)]
    for work in CRAWL_WORK_DIRS:
        cmd += ["--crawl-work-dir", str(work)]
    cmd += [
        "--",
        "--bbl-recovery-dir",
        str(bbl_dir),
        "--workers",
        str(WORKERS),
        "--shard-papers",
        str(SHARD_PAPERS),
        "--paper-timeout",
        str(PAPER_TIMEOUT_S),
        "--expect-assigned",
    ]
    if RUN_MODE == "smoke":
        cmd += ["--offset", str(SMOKE_OFFSET), "--limit", str(SMOKE_LIMIT)]
    proc = subprocess.Popen(cmd)
    last = time.monotonic()
    try:
        while proc.poll() is None:
            time.sleep(5)
            if time.monotonic() - last >= SYNC_EVERY_S:
                print(
                    "synced to Drive:",
                    sync_to_drive(OUT, WORK, REPORT, M2_DRIVE, include_report=False),
                )
                last = time.monotonic()
    finally:
        proc.wait()
        print(
            "exit code",
            proc.returncode,
            "; final sync:",
            sync_to_drive(OUT, WORK, REPORT, M2_DRIVE),
        )
    finish_session(session_state, session_id, proc.returncode)
    annotate_report(session_state, REPORT)
    print("synced measurement audit:", sync_to_drive(OUT, WORK, REPORT, M2_DRIVE))
    if proc.returncode:
        raise RuntimeError(f"the parser exited with {proc.returncode}: see {WORK / 'run.log'}")

## Check the tables and the tree

`check_tables` verifies types, unique IDs, parents and reachability on the whole output; `audit_structure` compares counts in the source with counts in the tree; `audit_sample` draws the 30 random papers (seed 2026) that a team member reads and scores.

In [ ]:
if RUN_M2 and (REPORT / "m2_benchmark_summary.csv").is_file():
    roots = [a for root in DATA_ROOTS for a in ("--data-root", str(root))]
    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.qa.check_tables",
            "--out-dir",
            str(OUT),
            "--json",
            str(REPORT / "check_tables.json"),
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.qa.audit_structure",
            *roots,
            "--out-dir",
            str(OUT),
            "--report",
            str(REPORT),
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.qa.audit_sample",
            "draw",
            *roots,
            "--out-dir",
            str(OUT),
            "--work-dir",
            str(WORK),
            "--report",
            str(REPORT / "audit30"),
            "-n",
            "30",
            "--seed",
            "2026",
        ],
        check=True,
    )
else:
    print("Skipped: no completed M2 benchmark session.")

## M1 statistics of the real crawl (the Book's `data/m1/`)

`src.analysis.stats` reads the five restored runs as one corpus and writes `papers_stats.csv`, `references_stats.csv` and `corpus_summary.csv` for the M1 chapters. Its timing files (`benchmark_summary.csv`, `measurement_audit.json`) report what the crawl sessions measured: the runs are partitions and their timing is **not certified** (`timing_complete` is false in every `coverage.json`), so those numbers are a lower bound, not the official M1 benchmark, and the Book says so.
The optimized M2 restore excludes M1 benchmark telemetry. Use the complete M1 checkpoints for any independent M1 export.


In [ ]:
if RUN_M1_STATS:
    raise RuntimeError(
        "M2 restore excludes M1 telemetry; run src.analysis.stats with full M1 checkpoints."
    )
print("M1 statistics export is disabled in the M2 notebook.")

## Export for the Book

`book_csv/` holds the files the Book reads (`data/m2/`). Everything is copied to Drive under `M2_DRIVE_DIR`.

In [ ]:
if RUN_M2 and (REPORT / "m2_benchmark_summary.csv").is_file():
    book_csv = REPORT / "book_csv"
    book_csv.mkdir(parents=True, exist_ok=True)
    for name in (
        "m2_benchmark_summary.csv",
        "m2_resource_timeseries.csv",
        "m2_papers.csv",
        "m2_versions.csv",
        "structure_audit.csv",
        "m2_sessions.jsonl",
        "measurement_audit.json",
    ):
        if (REPORT / name).is_file():
            shutil.copy2(REPORT / name, book_csv / name)
    for name in ("assigned_check.json", "run_summary.json"):
        if (WORK / name).is_file():
            shutil.copy2(WORK / name, REPORT / name)
    print("restore (setup) minutes, if this session did it:", round(RESTORE_SECONDS / 60, 1))
    print("final sync:", sync_to_drive(OUT, WORK, REPORT, M2_DRIVE))
    shutil.make_archive("/content/m2_evidence", "zip", REPORT)
    shutil.copy2("/content/m2_evidence.zip", M2_DRIVE / "m2_evidence.zip")
    print("Results on Drive:", M2_DRIVE)
else:
    print("Skipped: no completed M2 benchmark session.")